# 02 — YOLO11 + ByteTrack (pipeline cơ sở)

Nhóm 05 — Fall Detection. Bật **GPU**. Thêm dataset URFD & Le2i qua **Add Input**.


**Cài thư viện** (Kaggle/Colab) — bỏ qua nếu môi trường đã có sẵn.

In [ ]:
!pip -q install ultralytics lap

## Thư viện mã nguồn (nhúng sẵn trong notebook)
Toàn bộ hàm/lớp cần dùng được định nghĩa trong các cell dưới đây. Mỗi cell là một nhóm chức năng kèm mô tả; **chạy lần lượt từ trên xuống một lần** trước khi sang phần thực nghiệm.

**Tiện ích chung (seed, I/O, JSON)**

Cố định seed để tái lập (`set_seed`), kiểm tra môi trường Kaggle (`on_kaggle`) và các hàm I/O JSON.

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Tiện ích chung: seed tái lập, nạp config, I/O cơ bản."""
import os
import json
import random
from pathlib import Path

import numpy as np


def set_seed(seed: int = 42) -> None:
    """Cố định seed để kết quả tái lập được (yêu cầu nộp bài của thầy)."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


def load_config(path: str | Path) -> dict:
    """Nạp file YAML cấu hình."""
    import yaml
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def save_json(obj, path: str | Path) -> None:
    ensure_dir(Path(path).parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def load_json(path: str | Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def on_kaggle() -> bool:
    return os.path.exists("/kaggle/input")


**YOLO11 + ByteTrack (phát hiện + bám vết người)**

`Track` + `DetectorTracker`: chạy **YOLO11 phát hiện người + ByteTrack bám ID** trên một video; `write_mot` xuất kết quả chuẩn MOT.

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Pipeline dùng chung: YOLO11 (phát hiện người) + ByteTrack (gán ID).

Dùng tracker tích hợp của Ultralytics (model.track(..., tracker="bytetrack.yaml")).
Xuất kết quả chuẩn MOT để tính IDF1 và cho các mô hình phân loại dùng lại.
"""
from dataclasses import dataclass
from pathlib import Path


@dataclass
class Track:
    frame: int
    track_id: int
    x: float
    y: float
    w: float
    h: float
    conf: float


class DetectorTracker:
    def __init__(self, model_name: str = "yolo11n.pt", conf: float = 0.25, person_class: int = 0):
        from ultralytics import YOLO
        self.model = YOLO(model_name)
        self.conf = conf
        self.person_class = person_class

    def run(self, source, tracker: str = "bytetrack.yaml"):
        """Chạy detection + tracking trên 1 video, trả về list[Track].

        source: đường dẫn video (Le2i) hoặc list ảnh/thư mục (URFD).
        """
        tracks: list[Track] = []
        results = self.model.track(
            source=source, persist=True, tracker=tracker,
            classes=[self.person_class], conf=self.conf, stream=True, verbose=False,
        )
        for fi, r in enumerate(results):
            if r.boxes is None or r.boxes.id is None:
                continue
            for box, tid, cf in zip(r.boxes.xywh.cpu().numpy(),
                                    r.boxes.id.int().cpu().numpy(),
                                    r.boxes.conf.cpu().numpy()):
                x, y, w, h = box  # tâm + kích thước
                tracks.append(Track(fi, int(tid), float(x - w / 2), float(y - h / 2),
                                    float(w), float(h), float(cf)))
        return tracks


def write_mot(tracks: list[Track], out_path: str | Path) -> None:
    """Ghi file chuẩn MOT: frame,id,x,y,w,h,conf,-1,-1,-1 (phục vụ TrackEval)."""
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        for t in tracks:
            f.write(f"{t.frame + 1},{t.track_id},{t.x:.2f},{t.y:.2f},"
                    f"{t.w:.2f},{t.h:.2f},{t.conf:.3f},-1,-1,-1\n")


**Index dữ liệu URFD/Le2i + chia tập theo video**

Quét dữ liệu **URFD/Le2i** thành `VideoItem` (`index_urfd`, `index_le2i`, đọc annotation Le2i), chia tập **theo video/người** chống rò rỉ (`split_by_group`) và tạo các protocol within + cross-dataset (`build_splits`).

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Chia tập theo VIDEO (chống rò rỉ) và giao thức cross-dataset.

Nguyên tắc: không bao giờ chia theo frame — các frame liền kề gần giống nhau
sẽ rò rỉ giữa train/test. Mỗi video (và người thực hiện) chỉ thuộc một tập.
"""
import argparse
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Literal



@dataclass
class VideoItem:
    """Một đơn vị dữ liệu mức video/chuỗi."""
    vid: str                       # id duy nhất
    dataset: str                   # "urfd" | "le2i"
    label: int                     # 1 = có té ngã, 0 = ADL
    path: str
    subject: str | None = None     # người thực hiện (nếu biết) -> chia theo người
    scene: str | None = None       # bối cảnh/góc camera cho TN3
    fall_start: int | None = None  # frame bắt đầu ngã (nếu có nhãn)
    fall_end: int | None = None


_VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv"}


def index_urfd(root: str | Path) -> list[VideoItem]:
    """Quét thư mục URFD -> danh sách VideoItem (nhãn mức chuỗi).

    URFD gồm 70 chuỗi: fall-01..fall-30 (có té ngã) và adl-01..adl-40 (sinh hoạt
    thường). Mỗi chuỗi thường là một thư mục PNG (ví dụ 'fall-01-cam0-rgb') hoặc
    một file video. Hàm nhận diện nhãn qua tiền tố tên ('fall' -> 1, 'adl' -> 0)
    và camera qua 'cam0'/'cam1'. Dung nạp nhiều cách sắp xếp của các mirror Kaggle.

    URFD không cung cấp bbox người; fall_start/fall_end để None (ở mức sự kiện,
    cả chuỗi fall được coi là một sự kiện ngã — xem datasets.make_gt_falls).
    """
    root = Path(root)
    items: list[VideoItem] = []
    seen: set[str] = set()

    def classify(name: str) -> int | None:
        low = name.lower()
        if low.startswith("fall") or "-fall" in low or "/fall" in low:
            return 1
        if low.startswith("adl") or "-adl" in low or "/adl" in low:
            return 0
        return None

    # 1) Thư mục chuỗi PNG: */<seq>-camX-rgb/ chứa ảnh
    for d in sorted(root.rglob("*")):
        if not d.is_dir():
            continue
        if not any(d.glob("*.png")) and not any(d.glob("*.jpg")):
            continue
        label = classify(d.name)
        if label is None:
            continue
        cam = "cam1" if "cam1" in d.name.lower() else "cam0"
        vid = d.name
        if vid in seen:
            continue
        seen.add(vid)
        items.append(VideoItem(vid=vid, dataset="urfd", label=label, path=str(d),
                               subject=_seq_base(d.name), scene=cam))

    # 2) File video rời (một số mirror đóng gói thành .avi/.mp4)
    for f in sorted(root.rglob("*")):
        if f.suffix.lower() not in _VIDEO_EXTS:
            continue
        label = classify(f.stem)
        if label is None:
            continue
        cam = "cam1" if "cam1" in f.stem.lower() else "cam0"
        vid = f.stem
        if vid in seen:
            continue
        seen.add(vid)
        items.append(VideoItem(vid=vid, dataset="urfd", label=label, path=str(f),
                               subject=_seq_base(f.stem), scene=cam))

    if not items:
        raise FileNotFoundError(
            f"Không tìm thấy chuỗi URFD trong {root}. Kiểm tra lại đường dẫn/'Add Input' trên Kaggle.")
    return items


def _seq_base(name: str) -> str:
    """Lấy id chuỗi gốc (bỏ hậu tố cam/rgb) để chia tập không rò rỉ giữa 2 camera."""
    import re
    m = re.match(r"((?:fall|adl)[-_]?\d+)", name.lower())
    return m.group(1).replace("_", "-") if m else name


def parse_le2i_annotation(ann_path: Path):
    """Đọc annotation Le2i: 2 dòng đầu = frame bắt đầu/kết thúc ngã; các dòng sau =
    'frame,label,x1,y1,x2,y2' (bbox từng frame).

    Trả (fall_start, fall_end, bboxes) với bboxes = {frame: (x1,y1,x2,y2)}.
    Bao dung với biến thể định dạng (dấu phẩy/space/tab, thiếu bbox).
    """
    try:
        lines = [l.strip() for l in ann_path.read_text(errors="ignore").splitlines() if l.strip()]
    except OSError:
        return None, None, {}
    if not lines:
        return None, None, {}

    def ints(s):
        import re
        return [int(float(x)) for x in re.split(r"[,\s]+", s.strip()) if x != ""]

    fall_start = fall_end = None
    body_start = 0
    if len(lines) >= 2 and len(ints(lines[0])) == 1 and len(ints(lines[1])) == 1:
        fall_start, fall_end = ints(lines[0])[0], ints(lines[1])[0]
        body_start = 2

    bboxes: dict[int, tuple] = {}
    for l in lines[body_start:]:
        vals = ints(l)
        if len(vals) >= 6:
            fr, _lab, x1, y1, x2, y2 = vals[:6]
            if x2 > 0 or y2 > 0:
                bboxes[fr] = (x1, y1, x2, y2)
    if fall_start in (0, None) and fall_end in (0, None):
        fall_start = fall_end = None
    return fall_start, fall_end, bboxes


def index_le2i(root: str | Path) -> list[VideoItem]:
    """Quét Le2i (4 bối cảnh) + đọc annotation (frame ngã + bbox).

    Cấu trúc thường gặp: <root>/<Scene>/Videos/*.avi và <root>/<Scene>/Annotation_files/*.txt
    (hoặc annotation đặt cạnh video). Nhãn = 1 nếu annotation có mốc ngã, else 0.
    Scene lấy từ tên thư mục cha (Home/Coffee_room/Office/Lecture_room).
    """
    root = Path(root)
    items: list[VideoItem] = []
    for f in sorted(root.rglob("*")):
        if f.suffix.lower() not in _VIDEO_EXTS:
            continue
        ann = _find_le2i_annotation(f)
        fall_start = fall_end = None
        if ann is not None:
            fall_start, fall_end, _ = parse_le2i_annotation(ann)
        label = 1 if (fall_start is not None and fall_end is not None) else 0
        scene = _le2i_scene(f)
        items.append(VideoItem(vid=f"{scene}-{f.stem}", dataset="le2i", label=label,
                               path=str(f), subject=scene, scene=scene,
                               fall_start=fall_start, fall_end=fall_end))
    if not items:
        raise FileNotFoundError(
            f"Không tìm thấy video Le2i trong {root}. Kiểm tra lại đường dẫn/'Add Input' trên Kaggle.")
    return items


def _le2i_scene(video: Path) -> str:
    known = {"home", "coffee_room", "coffee", "office", "lecture_room", "lecture"}
    for p in video.parents:
        if p.name.lower().replace(" ", "_") in known:
            return p.name
    # thử thư mục cha cấp 1-2
    return video.parent.name


def _find_le2i_annotation(video: Path) -> Path | None:
    stem = video.stem
    cands = [
        video.with_suffix(".txt"),
        video.parent / f"{stem}.txt",
        video.parent.parent / "Annotation_files" / f"{stem}.txt",
        video.parent.parent / "Annotation" / f"{stem}.txt",
    ]
    for c in cands:
        if c.exists():
            return c
    # tìm mờ theo stem
    for c in video.parent.parent.rglob(f"{stem}.txt"):
        return c
    return None


def split_by_group(
    items: list[VideoItem],
    ratio=(0.70, 0.15, 0.15),
    key: Literal["video", "subject"] = "video",
    seed: int = 42,
) -> dict[str, list[str]]:
    """Chia train/val/test theo nhóm (video hoặc người), giữ cân bằng lớp thô."""
    import random
    rng = random.Random(seed)
    # gom theo key để không rò rỉ
    groups: dict[str, list[VideoItem]] = {}
    for it in items:
        gkey = getattr(it, "vid" if key == "video" else key, None) or it.vid
        groups.setdefault(gkey, []).append(it)
    keys = list(groups)
    rng.shuffle(keys)
    n = len(keys)
    n_tr = int(ratio[0] * n)
    n_va = int(ratio[1] * n)
    buckets = {"train": keys[:n_tr], "val": keys[n_tr:n_tr + n_va], "test": keys[n_tr + n_va:]}
    return {s: [v for k in ks for v in [it.vid for it in groups[k]]] for s, ks in buckets.items()}


def build_splits(cfg: dict) -> dict:
    """Tạo cả split within-dataset và 2 protocol cross-dataset."""
    set_seed(cfg["seed"])
    urfd = index_urfd(cfg["paths"]["urfd_root"])
    le2i = index_le2i(cfg["paths"]["le2i_root"])
    ratio = cfg["data"]["split_ratio"]

    out = {
        "within_urfd": split_by_group(urfd, ratio, seed=cfg["seed"]),
        "within_le2i": split_by_group(le2i, ratio, seed=cfg["seed"]),
        # Protocol A: train URFD -> test Le2i ; Protocol B: train Le2i -> test URFD
        "cross_A_train_urfd_test_le2i": {
            "train": [it.vid for it in urfd],
            "test": [it.vid for it in le2i],
        },
        "cross_B_train_le2i_test_urfd": {
            "train": [it.vid for it in le2i],
            "test": [it.vid for it in urfd],
        },
    }
    # lưu cả metadata từng video để dùng lại
    out["_items"] = [asdict(it) for it in (urfd + le2i)]
    save_json(out, Path(cfg["paths"]["splits_dir"]) / "splits.json")
    return out


## Cấu hình & chạy thực nghiệm

In [ ]:
# === Cấu hình (nhúng inline — không cần configs/default.yaml hay src/) ===
cfg = {
    "seed": 42,
    "paths": {
        "urfd_root": "data/urfd",     # TRÊN KAGGLE: /kaggle/input/<slug>/...
        "le2i_root": "data/le2i",     # TRÊN KAGGLE: /kaggle/input/<slug>/...
        "work_dir":  "outputs",       # /kaggle/working khi chạy Kaggle
        "splits_dir": "splits",
    },
    "data": {
        "fps_sample": 5, "img_size": 224, "seq_len": 16,
        "split_ratio": [0.70, 0.15, 0.15], "split_by": "video",
    },
    "detection": {"model": "yolo11n.pt", "conf": 0.25, "person_class": 0},
    "tracking":  {"type": "bytetrack", "track_thresh": 0.5, "match_thresh": 0.8},
    "evaluate":  {"tolerance_sec": 1.0},
    "model2_pose_lstm": {"hidden": 128, "layers": 1, "dropout": 0.3, "lr": 0.001, "epochs": 40},
    "model3_cnn_gru":   {"backbone": "mobilenet_v2", "hidden": 128, "dropout": 0.4, "lr": 0.001, "epochs": 30},
}
set_seed(cfg["seed"])
# TRÊN KAGGLE: trỏ cfg["paths"] tới dataset đã Add Input, ví dụ:
# cfg["paths"]["urfd_root"] = "/kaggle/input/urfd/..."
# cfg["paths"]["le2i_root"] = "/kaggle/input/le2i/..."
print("Kaggle:", on_kaggle())


In [ ]:
dt = DetectorTracker(cfg['detection']['model'], cfg['detection']['conf'])
le2i = index_le2i(cfg['paths']['le2i_root'])
tracks = dt.run(le2i[0].path)
write_mot(tracks, 'outputs/mot/'+le2i[0].vid+'.txt')
print('Số track-frame:', len(tracks))

## TN1 — YOLO11n vs YOLO11s (mAP@0.5 trên Le2i có bbox + FPS)

In [ ]:
# Tạo data.yaml từ bbox Le2i rồi dùng YOLO.val() cho từng cỡ mô hình.
# from ultralytics import YOLO; import time
# for m in ['yolo11n.pt','yolo11s.pt']:
#     metrics = YOLO(m).val(data='le2i_person.yaml')  # -> mAP@0.5
#     ... đo FPS bằng time trên cùng tập
print('TODO TN1: chuẩn bị le2i_person.yaml từ parse_le2i_annotation (bbox).')